In [1]:
import pandas as pd
import numpy as np
import os
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
import pickle
from sklearn.preprocessing import StandardScaler, RobustScaler

# Suppress warnings to keep the output clean during training
warnings.filterwarnings('ignore')


In [2]:
def load_and_preprocess_data(path='../dataset/', target_variables=None, modis_data_dir=None, apply_scaling=True):
    """
    Loads the datasets and performs comprehensive preprocessing steps,
    including merging, handling missing values (NaN and Inf), and scaling.

    Args:
        path (str): The base directory where the dataset files are located.
        target_variables (list): List of target variable column names to exclude from preprocessing (e.g., from scaling).
        modis_data_dir (str, optional): Path to the directory containing processed MODIS/Landsat/Sentinel parquet files.
                                        If None, these additional datasets are not loaded.
        apply_scaling (bool): Whether to apply StandardScaler to numerical features.

    Returns:
        tuple: A tuple containing preprocessed train_df, test_df, train_gap_df, and test_gap_df.
    """
    print("--- Loading and Preprocessing Data ---")

    # Initialize stage_transformers to store preprocessing information
    stage_transformers = {
        'imputation_means': {}, # Stores means from training data for consistent imputation
        'scaler': None,
        'scaled_columns': [],
    }

    # Load main datasets
    train_path = os.path.join(path, 'Train.csv')
    test_path = os.path.join(path, 'Test.csv')
    gap_train_path = os.path.join(path, 'Gap_Train.csv')
    gap_test_path = os.path.join(path, 'Gap_Test.csv')

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    train_gap_df = pd.read_csv(gap_train_path)
    test_gap_df = pd.read_csv(gap_test_path)

    # Merge BulkDensity into test_gap_df, crucial for later calculations
    test_gap_df = pd.merge(test_gap_df, test_df[['PID', 'BulkDensity']], on='PID', how='left')

    # Handle missing values in initial train/test CSVs with mean imputation
    print("Handling initial missing values (mean imputation)...")
    for column in train_df.columns:
        if train_df[column].isnull().any():
            mean_value = train_df[column].mean()
            train_df[column].fillna(mean_value, inplace=True)
            stage_transformers['imputation_means'][column] = mean_value

    for column in test_df.columns:
        if test_df[column].isnull().any():
            fill_value = stage_transformers['imputation_means'].get(column, test_df[column].mean()) # Use train mean or test mean
            test_df[column].fillna(fill_value, inplace=True)
    print("Initial missing values handled.")

    # --- Load and process multiple auxiliary datasets from parquet files ---
    if modis_data_dir:
        print(f"Loading additional data from {modis_data_dir}...")
        # Dictionary mapping product names to their columns that should be aggregated
        # The keys are the actual product identifiers (e.g., 'MCD43A4', 'L8', 'S1', 'S2')
        # The 'prefix' is used to construct the filename (e.g., 'modis', 'landsat', 'sentinel')
        auxiliary_products_info = {
            '_MCD43A4': {
                'prefix': 'modis',
                'cols': ['NDVI_mcd43a4', 'EVI_mcd43a4', 'SAVI_mcd43a4', 'GNDVI_mcd43a4',
                         'SR_mcd43a4', 'NDBR_mcd43a4', 'GRVI_mcd43a4', 'Brightness_Index_mcd43a4',
                         'Red_Green_Ratio_mcd43a4', 'Chlorophyll_Index_mcd43a4', 'Blue_NIR_Ratio_mcd43a4']
            },
            '_MOD09GA': {
                'prefix': 'modis',
                'cols': ['sur_refl_b01', 'sur_refl_b02', 'sur_refl_b03', 'sur_refl_b04', 'sur_refl_b05',
                         'sur_refl_b06', 'sur_refl_b07', 'NDVI_mod09ga', 'EVI_mod09ga', 'SAVI_mod09ga',
                         'NDWI_mod09ga', 'BSI_mod09ga', 'GEMI_mod09ga', 'ARVI_mod09ga', 'SIPI_mod09ga']
            },
            '_MOD11A1': {
                'prefix': 'modis',
                'cols': ['LST_Day_1km', 'LST_Night_1km']
            },
            '_MOD13Q1': {
                'prefix': 'modis',
                'cols': ['season_sin', 'season_cos', 'EVI_scaled', 'NDVI_scaled', 'SAVI_m13q1',
                         'MSAVI_m13q1', 'SR_m13q1', 'NDBR_m13q1', 'NDSWIR_m13q1', 'NDSWIR_NIR_m13q1',
                         'Brightness_Index_m13q1', 'Red_Blue_Ratio_m13q1', 'SWIR_Blue_Ratio_m13q1',
                         'Chlorophyll_Red_Edge_m13q1', 'NRI_approx_m13q1', 'PSRI_m13q1', 'SIPI_m13q1', 'MSI_m13q1']
            },
            '_MOD16A2': {
                'prefix': 'modis',
                'cols': ['ET', 'PET', 'ESI_mod16a2', 'ETD_mod16a2']
            },
            '8': { # Changed from '8' to 'L8' for clarity and consistent naming
                'prefix': 'landsat',
                'cols': ['LST_Celsius', 'NDVI_ls8', 'EVI_ls8', 'SAVI_ls8', 'NDWI_ls8', 'BSI_ls']
            },
            '1': { # Changed from '1' to 'S1'
                'prefix': 'sentinel',
                'cols': ['VH_to_VV_Ratio_dB', 'VV_minus_VH_dB', 'Span_dB']
            },
            '2': { # Changed from '2' to 'S2'
                'prefix': 'sentinel',
                'cols': ['NDVI_sent2', 'EVI_sent2', 'SAVI_sent2', 'NDRE1_sent2', 'CIre_sent2',
                         'NDWI_sent2', 'LSWI_sent2', 'BSI_sent2', 'NDRE2_sent2',
                         'CLOUDY_PIXEL_PERCENTAGE', 'NODATA_PIXEL_PERCENTAGE']
            }
        }

        for product_name, info in auxiliary_products_info.items():
            file_prefix = info['prefix']
            cols_to_aggregate = info['cols']

            # Construct the parquet filename using the correct format: processed_prefix_productname.parquet
            file_path = os.path.join(modis_data_dir, f'processed_{file_prefix}{product_name.lower()}.parquet')

            try:
                # Load parquet file
                product_df = pd.read_parquet(file_path)

                if 'PID' not in product_df.columns:
                    print(f"Warning: 'PID' column not found in {product_name} data ({file_path}). Skipping merge for this product.")
                    continue

                valid_cols = [col for col in cols_to_aggregate if col in product_df.columns]
                if not valid_cols:
                    print(f"Warning: No valid columns to aggregate found in {product_name} ({file_path}). Skipping merge for this product.")
                    continue

                product_aggregated = product_df.groupby('PID')[valid_cols].mean().reset_index()
                rename_dict = {col: f'{product_name}_{col}' for col in valid_cols}
                product_aggregated.rename(columns=rename_dict, inplace=True)

                train_df = pd.merge(train_df, product_aggregated, on='PID', how='left')
                test_df = pd.merge(test_df, product_aggregated, on='PID', how='left')

                # --- Robust NaN handling for newly merged columns ---
                for col_orig, col_renamed in rename_dict.items():
                    if col_renamed in train_df.columns:
                        if train_df[col_renamed].isnull().any():
                            mean_val_merged = train_df[col_renamed].mean()
                            train_df[col_renamed].fillna(mean_val_merged, inplace=True)
                            stage_transformers['imputation_means'][col_renamed] = mean_val_merged
                        if test_df[col_renamed].isnull().any():
                            fill_val_merged = stage_transformers['imputation_means'].get(col_renamed, test_df[col_renamed].mean())
                            test_df[col_renamed].fillna(fill_val_merged, inplace=True)
                print(f"{product_name} data loaded and merged successfully from {file_path}.")

            except FileNotFoundError:
                print(f"Error: {product_name} file not found at {file_path}. Skipping this product.")
            except Exception as e:
                print(f"An error occurred while processing {product_name} data from {file_path}: {e}. Skipping this product.")
    else:
        print("Auxiliary data directory not provided. Skipping loading of these datasets.")

    # Set default target variables if not provided
    if target_variables is None:
        target_variables = []

    # Identify numerical columns for final NaN/Inf handling and scaling
    numerical_cols_for_final_prep = [col for col in train_df.columns
                                     if train_df[col].dtype in ['int64', 'float64']
                                     and col != 'PID' and col != 'site'
                                     and col not in target_variables]

    print("\nPerforming final check and robust imputation for NaN/Inf values...")
    for col in numerical_cols_for_final_prep:
        # Convert any +/- inf to NaN
        if (np.isinf(train_df[col]).any()):
            train_df[col].replace([np.inf, -np.inf], np.nan, inplace=True)
            print(f"  Replaced Inf values in training column '{col}' with NaN.")
        if (np.isinf(test_df[col]).any()):
            test_df[col].replace([np.inf, -np.inf], np.nan, inplace=True)
            print(f"  Replaced Inf values in testing column '{col}' with NaN.")

        # Final imputation for any remaining NaNs (including those from Inf conversion)
        if train_df[col].isnull().any():
            final_mean = train_df[col].mean()
            # If a column becomes all NaN (e.g., from an empty merge or all Inf values), mean will be NaN.
            # In that edge case, fill with 0 to ensure no NaNs.
            if np.isnan(final_mean):
                final_mean = 0
                print(f"  Warning: Column '{col}' in train_df is entirely NaN after processing. Filling with 0.")
            train_df[col].fillna(final_mean, inplace=True)
            stage_transformers['imputation_means'][col] = final_mean # Update this mean for test_df consistency

        if test_df[col].isnull().any():
            # Use the mean from the training set, or 0 if training mean was NaN
            fill_val = stage_transformers['imputation_means'].get(col, 0)
            test_df[col].fillna(fill_val, inplace=True)

    print("Final NaN/Inf check and imputation complete.")


    # Feature Scaling
    if apply_scaling and numerical_cols_for_final_prep:
        print("Applying StandardScaler to numerical features...")

        cols_to_scale = numerical_cols_for_final_prep # All numerical features after robust imputation

        if cols_to_scale:
            scaler = StandardScaler()

            # Fit scaler on training data for selected numerical columns
            train_df[cols_to_scale] = scaler.fit_transform(train_df[cols_to_scale])

            # Transform test data using the same scaler
            test_df[cols_to_scale] = scaler.transform(test_df[cols_to_scale])

            # Store scaler and column names for future use
            stage_transformers['scaler'] = scaler
            stage_transformers['scaled_columns'] = cols_to_scale
            print(f"Scaled {len(cols_to_scale)} numerical features: {cols_to_scale}")
        else:
            print("No numerical features available for scaling.")
            stage_transformers['scaler'] = None
            stage_transformers['scaled_columns'] = []
    elif apply_scaling and not numerical_cols_for_final_prep:
        print("No numerical features found for scaling.")
        stage_transformers['scaler'] = None
        stage_transformers['scaled_columns'] = []
    else:
        print("Scaling not applied as per 'apply_scaling=False'.")


    print("Data loading and preprocessing complete.")
    print(f"Final train_df shape: {train_df.shape}")
    print(f"Final test_df shape: {test_df.shape}")

    return train_df, test_df, train_gap_df, test_gap_df



In [3]:
def train_random_forest_flexible_features(train_df, test_df, target_columns, feature_list=None, models_dir='rf_flexible_models', model_name='rf_base.pkl'):
    """
    Trains a RandomForestRegressor (wrapped in MultiOutputRegressor) with n_estimators=500
    using a flexible set of features. By default, it uses a predefined list of
    'core features', but can be updated with any custom list of features.
    No hyperparameter tuning is performed in this function.

    Args:
        train_df (pd.DataFrame): The preprocessed training DataFrame.
        test_df (pd.DataFrame): The preprocessed test DataFrame.
        target_columns (list): A list of target column names (e.g., ['N', 'P', ...]).
        feature_list (list, optional): A list of feature column names to use for training.
                                        If None, a default list of 'core features' will be used.
        models_dir (str): Directory to save the trained RandomForest model.

    Returns:
        tuple: A tuple containing:
            - model (MultiOutputRegressor): The trained RandomForest model.
            - rmse (float): RMSE score on the validation set.
            - predictions (np.ndarray): Predictions on the X_test_final data.
    """
    print("\n--- Starting RandomForest Model Training with Flexible Features ---")

    # Create directory for saving the model if it doesn't exist
    if not os.path.exists(models_dir):
        os.makedirs(models_dir)

    # Define the default core features if no feature_list is provided
    if feature_list is None:
        selected_features = [
            'pH', 'ph20', 'BulkDensity', 'cec20', 'ecec20', 'hp20',
            'snd20', 'soc20', 'xhp20'
        ]
        print("Using default 'Core Features' for training.")
    else:
        selected_features = feature_list
        print(f"Using custom feature list for training.")


    # Validate that all selected_features exist in train_df and test_df
    missing_train_cols = [col for col in selected_features if col not in train_df.columns]
    missing_test_cols = [col for col in selected_features if col not in test_df.columns]

    if missing_train_cols:
        raise ValueError(f"Missing core features in train_df: {missing_train_cols}")
    if missing_test_cols:
        raise ValueError(f"Missing core features in test_df: {missing_test_cols}")

    # Feature selection for training data
    X = train_df[selected_features]
    y = train_df[target_columns]

    # Feature selection for test data.
    # It's crucial that test features match the training features used.
    X_test_final = test_df[selected_features]

    print(f"Features selected: {selected_features}")
    print(f"X_train shape (after feature selection): {X.shape}")
    print(f"X_test_final shape (after feature selection): {X_test_final.shape}")

    # Train-validation split
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=17)
    print(f"Data split into training and validation sets: Train({X_train.shape}), Val({X_val.shape})")

    # Define and train the RandomForestRegressor model
    # n_estimators is 450 as requested, default params for others.
    # n_jobs=-1 utilizes all available CPU cores for parallel processing.
    base_rf_model = LGBMRegressor(n_estimators=450, learning_rate=0.03, random_state=17, verbose=-1, n_jobs=-1)
    model = MultiOutputRegressor(base_rf_model)

    print("Training LGBMRegressor model...")
    model.fit(X_train, y_train)
    print("LGBMRegressor training complete.")

    # Evaluate the model on the validation set
    y_val_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    print(f"LGBMRegressor Validation RMSE: {rmse:.4f}")

    final_model = MultiOutputRegressor(base_rf_model)
    final_model.fit(X, y)

    # Make predictions on the final test set
    predictions = final_model.predict(X_test_final)
    print("Predictions made on test data.")

    # Save the trained model
    model_filename = os.path.join(models_dir, model_name)
    with open(model_filename, 'wb') as file:
        pickle.dump(final_model, file)
    print(f"RandomForest model saved to {model_filename}")

    print("--- RandomForest Model Training with Flexible Features Complete ---")
    return model, rmse, predictions


In [4]:
def create_submission_file(test_df, predictions, test_gap_df, target_columns, output_filename='submission.csv'):
    """
    Generates the final submission CSV file in the required format.
    It takes the raw model predictions and transforms them into 'ID' and 'Gap' columns.

    Args:
        test_df (pd.DataFrame): The original test DataFrame (used primarily for 'PID').
        predictions (np.ndarray): The raw predictions from the model (e.g., from the VotingRegressor).
        test_gap_df (pd.DataFrame): The preprocessed test_gap_df, which should contain
                                     'Required' and 'BulkDensity' columns.
        target_columns (list): A list of target column names (e.g., ['N', 'P', 'K', ...]).
        output_filename (str): The desired name for the output CSV submission file.
    """
    print(f"\n--- Creating Submission File: {output_filename} ---")

    # Basic check to ensure predictions align with expected structure
    if predictions.shape[1] != len(target_columns):
        print(f"Warning: Prediction array columns ({predictions.shape[1]}) do not match target column count ({len(target_columns)}).")
    if predictions.shape[0] != test_df.shape[0]:
        print(f"Warning: Prediction array rows ({predictions.shape[0]}) do not match test_df rows ({test_df.shape[0]}).")

    # Convert raw predictions into a pandas DataFrame, associating them with PIDs
    predictions_df = pd.DataFrame(predictions, columns=target_columns)
    predictions_df['PID'] = test_df['PID']

    # Melt the predictions_df from wide format to long format.
    # This creates columns for 'PID', 'Nutrient' (e.g., 'N', 'P'), and 'Available_Nutrients_in_ppm'.
    submission_melted = predictions_df.melt(
        id_vars=['PID'],
        var_name='Nutrient',
        value_name='Available_Nutrients_in_ppm'
    )
    # Sort by PID for consistency and reset index
    submission_melted = submission_melted.sort_values('PID').reset_index(drop=True)

    # Merge the melted predictions with the test_gap_df.
    # This brings in the 'Required' nutrient levels and 'BulkDensity' needed for calculations.
    nutrient_df = pd.merge(test_gap_df, submission_melted, on=['PID', 'Nutrient'], how='left')

    # Calculate 'Available_Nutrients_in_kg_ha' based on the provided formula.
    # soil_depth is constant at 20 cm as per the notebook.
    soil_depth = 20  # cm
    nutrient_df['Available_Nutrients_in_kg_ha'] = (
        nutrient_df['Available_Nutrients_in_ppm'] * soil_depth * nutrient_df['BulkDensity'] * 0.1
    )

    # Calculate the 'Gap' which is the difference between 'Required' and 'Available'.
    # A positive gap means the nutrient needs to be added, negative means there's an excess.
    nutrient_df['Gap'] = nutrient_df['Required'] - nutrient_df['Available_Nutrients_in_kg_ha']

    # Create the unique 'ID' column by concatenating 'PID' and 'Nutrient'.
    nutrient_df['ID'] = nutrient_df['PID'].astype(str) + "_" + nutrient_df['Nutrient']

    # Select only the 'ID' and 'Gap' columns for the final submission file.
    final_submission_df = nutrient_df[['ID', 'Gap']]

    # Save the final DataFrame to a CSV file without the DataFrame index.
    final_submission_df.to_csv(output_filename, index=False)
    print(f"Submission file saved successfully as {output_filename}")
    print(f"First 5 rows of the generated submission file:\n", final_submission_df.head())
    print("--- Submission File Creation Complete ---")


In [6]:
# --- Example of how to use the new train_random_forest_flexible_features function ---
if __name__ == "__main__":
    # Define the path to your dataset
    DATASET_PATH = '../dataset/'
    TARGET_COLUMNS = ['N', 'P', 'K', 'Ca', 'Mg', 'S', 'Fe', 'Mn', 'Zn', 'Cu', 'B']

    # Step 1: Load and preprocess data (assuming this function is already in your notebook)
    train_df, test_df, train_gap_df, test_gap_df = load_and_preprocess_data(DATASET_PATH, target_variables=TARGET_COLUMNS, modis_data_dir='../processed_data/', apply_scaling=True)

    # --- Scenario 1: Using the default "Core Features" ---
    print("\n--- Running Scenario 1: Default Core Features ---")
    rf_default_model, rf_default_rmse, rf_default_predictions = \
        train_random_forest_flexible_features(
            train_df, test_df, TARGET_COLUMNS,
            models_dir='rf_flexible_models_default_output',
            model_name='rf_default_features.pkl'  # Save the model with a specific name
        )
    create_submission_file(
        test_df, rf_default_predictions, test_gap_df, TARGET_COLUMNS,
        output_filename='submission_rf_default_features.csv'
    )
    print("\nScenario 1 complete.")

    # --- Scenario 2: Using a custom set of features (e.g., adding bio1 and bio12) ---
    print("\n--- Running Scenario 2: Custom Feature List (Core + Topographical) ---")
    custom_features = [
        'pH', 'ph20', 'BulkDensity', 'cec20', 'ecec20', 'hp20',
        'snd20', 'soc20', 'xhp20', # Core features
        'mdem', 'slope', 'tim'          # Added topographical features
    ]
    rf_custom_model, rf_custom_rmse, rf_custom_predictions = \
        train_random_forest_flexible_features(
            train_df, test_df, TARGET_COLUMNS,
            feature_list=custom_features, # Pass your custom list here
            model_name='rf_topographical_features.pkl'
        )
    create_submission_file(
        test_df, rf_custom_predictions, test_gap_df, TARGET_COLUMNS,
        output_filename='submission_rf_+_topo.csv'
    )
    print("\nScenario 2 complete.")

    # --- Scenario 3: Using a custom set of features (e.g., adding bio1 and bio12) ---
    print("\n--- Running Scenario 3: Custom Feature List (Core + Topographical + Climate variables) ---")
    custom_features = [
        'pH', 'ph20', 'BulkDensity', 'cec20', 'ecec20', 'hp20',
        'snd20', 'soc20', 'xhp20', # Core features
        'mdem', 'slope', 'tim',          # Added topographical features
        'bio1', 'bio12', 'bio15', 'bio7',
        'lstd', 'lstn'                                    # Added climate features
    ]
    rf_custom_model, rf_custom_rmse, rf_custom_predictions = \
        train_random_forest_flexible_features(
            train_df, test_df, TARGET_COLUMNS,
            feature_list=custom_features, # Pass your custom list here
            model_name='rf_topo_climate_features.pkl'
        )
    create_submission_file(
        test_df, rf_custom_predictions, test_gap_df, TARGET_COLUMNS,
        output_filename='submission_rf_+_climate.csv'
    )
    print("\nScenario 3 complete.")

        # --- Scenario 4: Using a custom set of features (e.g., adding bio1 and bio12) ---
    print("\n--- Running Scenario 4: Custom Feature List (Core + Topographical + Climate variables) + Water Variables ---")
    custom_features = [
        'pH', 'ph20', 'BulkDensity', 'cec20', 'ecec20', 'hp20',
        'snd20', 'soc20', 'xhp20', # Core features
        'mdem', 'slope', 'tim',          # Added topographical features
        'bio1', 'bio12', 'bio15', 'bio7',
        'lstd', 'lstn',                                    # Added climate features
        'dows', 'ls'                                       # Added water variables
    ]
    rf_custom_model, rf_custom_rmse, rf_custom_predictions = \
        train_random_forest_flexible_features(
            train_df, test_df, TARGET_COLUMNS,
            feature_list=custom_features, # Pass your custom list here
            model_name='rf_topo_water_features.pkl'
        )
    create_submission_file(
        test_df, rf_custom_predictions, test_gap_df, TARGET_COLUMNS,
        output_filename='submission_rf_+_water.csv'
    )
    print("\nScenario 4 complete.")

            # --- Scenario 5: Using a custom set of features (e.g., adding bio1 and bio12) ---
    print("\n--- Running Scenario 5: Custom Feature List (Core + Topographical + Climate variables) + Water Variables + Land Cover ---")
    custom_features = [
        'pH', 'ph20', 'BulkDensity', 'cec20', 'ecec20', 'hp20',
        'snd20', 'soc20', 'xhp20', # Core features
        'mdem', 'slope', 'tim',          # Added topographical features
        'bio1', 'bio12', 'bio15', 'bio7',
        'lstd', 'lstn',                                    # Added climate features
        'dows', 'ls',                                       # Added water variables
        'alb', 'wp'                                        # Added land cover variables
    ]
    rf_custom_model, rf_custom_rmse, rf_custom_predictions = \
        train_random_forest_flexible_features(
            train_df, test_df, TARGET_COLUMNS,
            feature_list=custom_features, # Pass your custom list here
            model_name='rf_topo_land_cover_features.pkl'
        )
    create_submission_file(
        test_df, rf_custom_predictions, test_gap_df, TARGET_COLUMNS,
        output_filename='submission_rf_+_land_cover.csv'
    )
    print("\nScenario 5 complete.")

    

                # --- Scenario 6: Using a custom set of features (e.g., adding bio1 and bio12) ---
    print("\n--- Running Scenario 6: Custom Feature List (Core + Topographical + Climate variables) + Water Variables + Land Cover + mod16a2 ---")
    modis16a2_cols = ['_MOD16A2_ET', '_MOD16A2_PET', '_MOD16A2_ESI_mod16a2', '_MOD16A2_ETD_mod16a2']
    custom_features = custom_features + modis16a2_cols  # Add MODIS16A2 features to the custom list
    rf_custom_model, rf_custom_rmse, rf_custom_predictions = \
        train_random_forest_flexible_features(
            train_df, test_df, TARGET_COLUMNS,
            feature_list=custom_features, # Pass your custom list here
            model_name='rf_modis16a2.pkl'
        )
    create_submission_file(
        test_df, rf_custom_predictions, test_gap_df, TARGET_COLUMNS,
        output_filename='submission_rf_+_modis16a2.csv'
    )
    print("\nScenario 6 complete.")


                # --- Scenario 7: Using a custom set of features (e.g., adding bio1 and bio12) ---
    print("\n--- Running Scenario 7: Custom Feature List (Adding landsat8 features) ---")
    landsat_cols = ['8_LST_Celsius', '8_NDVI_ls8', '8_EVI_ls8', '8_SAVI_ls8', '8_NDWI_ls8', '8_BSI_ls']
    custom_features = custom_features + landsat_cols  # Add Landsat features to the custom list
    rf_custom_model, rf_custom_rmse, rf_custom_predictions = \
        train_random_forest_flexible_features(
            train_df, test_df, TARGET_COLUMNS,
            feature_list=custom_features, # Pass your custom list here
            model_name='rf_landsat.pkl'
        )
    create_submission_file(
        test_df, rf_custom_predictions, test_gap_df, TARGET_COLUMNS,
        output_filename='submission_rf_+_landsat.csv'
    )
    print("\nScenario 7 complete.")

    print("\nOverall script execution complete with flexible RandomForest training.")
    print("Check respective directories for saved models and submission files.")



--- Loading and Preprocessing Data ---
Handling initial missing values (mean imputation)...
Initial missing values handled.
Loading additional data from ../processed_data/...
_MCD43A4 data loaded and merged successfully from ../processed_data/processed_modis_mcd43a4.parquet.
_MOD09GA data loaded and merged successfully from ../processed_data/processed_modis_mod09ga.parquet.
_MOD13Q1 data loaded and merged successfully from ../processed_data/processed_modis_mod13q1.parquet.
_MOD16A2 data loaded and merged successfully from ../processed_data/processed_modis_mod16a2.parquet.
8 data loaded and merged successfully from ../processed_data/processed_landsat8.parquet.
1 data loaded and merged successfully from ../processed_data/processed_sentinel1.parquet.
2 data loaded and merged successfully from ../processed_data/processed_sentinel2.parquet.

Performing final check and robust imputation for NaN/Inf values...
  Replaced Inf values in training column '_MOD09GA_SIPI_mod09ga' with NaN.
  Replace

ValueError: Missing core features in train_df: ['8_BSI_ls']

In [ ]:
train_df.columns[80:90]

Index(['_MOD13Q1_MSI_m13q1', '_MOD16A2_ET', '_MOD16A2_PET',
       '_MOD16A2_ESI_mod16a2', '_MOD16A2_ETD_mod16a2', '8_LST_Celsius',
       '8_NDVI_ls8', '8_EVI_ls8', '8_SAVI_ls8', '8_NDWI_ls8'],
      dtype='object')